# Counterfactual phenotype traversal violins (Figure 4)

Morphometric readout of the DiffAE counterfactual traversal for three example targets, one metric each:
**POLR1B** (nucleolar circularity), **TIM23 complex** (mitochondrial network degree), and **mTOR**
(lysosome radial position). Each panel shows real NTC, real KO, and the generated traversal at
α = 0, 1, 2, 3 (α=0 = DDIM-inverted reconstruction of the real NTC anchor; α=1 ≈ the KO mean; α>1
extrapolates beyond it).

**Normalization.** Every value is a %-change against a single shared baseline — the **real-NTC mean** for
that feature — applied identically to real NTC, real KO, and every generated α:

```
pct(x) = (x - mean(real_ntc)) / |mean(real_ntc)| * 100
```

Real NTC therefore sits at 0% by construction; real KO and each generated α show the measured effect size
on the same scale, so all three panels (different native units — circularity is unitless, degree is
unitless, radial position is a normalized 0–1 fraction) are directly comparable.

Inputs are the per-cell measured values (native units, pre-normalization), read from
`../../../data/figures/figure_4/`:

| file | target | feature |
| --- | --- | --- |
| `figure_4l-n_polr1b_circularity_traversal.csv` | POLR1B (nucleolus, phase) | circularity (4π·area/perimeter²) |
| `figure_4l-n_tim23_degree_traversal.csv` | TIM23 complex (mitochondria) | mean network degree |
| `figure_4l-n_mtor_location_traversal.csv` | mTOR (lysosome) | radial position (0 = nucleus, 1 = cell edge) |

Each CSV is long-format: `series` (`real_ntc`, `real_ko`, `gen_alpha0..3`) × `value` (one row per cell).

Per-series cell counts are unequal and vary by target (e.g. POLR1B real_ntc n=98 vs gen_alpha3 n=75; TIM23 real_ntc n=1000 vs each generated α n=100), so violin widths are not comparable across series — see the summary table for n.


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter

# Keep text editable in Illustrator (SVG keeps <text> elements rather than
# path-tracing the glyphs; PDF uses TrueType instead of Type-3).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data paths

Per-cell traversal CSVs, read from the central figure-data directory
`../../../data/figures/figure_4/`.


In [ ]:
FIGURE_DATA = Path("../../../data/figures/figure_4")

FEATURES = [
    dict(key="polr1b", csv="figure_4l-n_polr1b_circularity_traversal.csv",
         title="POLR1B — nucleolar circularity", ylabel="circularity"),
    dict(key="tim23", csv="figure_4l-n_tim23_degree_traversal.csv",
         title="TIM23 complex — mitochondrial network degree", ylabel="network degree"),
    dict(key="mtor", csv="figure_4l-n_mtor_location_traversal.csv",
         title="mTOR — lysosome radial position", ylabel="radial position"),
]


## Configuration

In [ ]:
SERIES = ["real_ntc", "real_ko", "gen_alpha0", "gen_alpha1", "gen_alpha2", "gen_alpha3"]
LABELS = ["real\nNTC", "real\nKO", "\u03b1=0", "\u03b1=1", "\u03b1=2", "\u03b1=3"]

COLORS = {
    "real_ntc": "#999999", "real_ko": "#2e8b57",
    "gen_alpha0": "#c6dbef", "gen_alpha1": "#6baed6",
    "gen_alpha2": "#3182bd", "gen_alpha3": "#08519c",
}

## Load

In [ ]:
def load(csv_name: str) -> dict[str, np.ndarray]:
    df = pd.read_csv(FIGURE_DATA / csv_name)
    return {s: df.loc[df["series"] == s, "value"].to_numpy() for s in SERIES}

data = {f["key"]: load(f["csv"]) for f in FEATURES}
for f in FEATURES:
    counts = {s: len(data[f["key"]][s]) for s in SERIES}
    print(f"{f['key']}: {counts}")

## Figure

One panel per target; violins show a black median line only (no extrema/whisker). Values are %-change
vs. the real-NTC mean (see Normalization above), so real NTC sits at 0% in every panel and the three
panels — different native units — are on one comparable axis.

In [ ]:
def pct(vals: np.ndarray, base: float) -> np.ndarray:
    return (vals - base) / abs(base) * 100.0


def plot_panel(ax, feat_data: dict[str, np.ndarray], title: str):
    base = float(np.mean(feat_data["real_ntc"]))
    pct_data = [pct(feat_data[s], base) for s in SERIES]

    vp = ax.violinplot(pct_data, positions=range(len(SERIES)), widths=0.8,
                       showmedians=True, showextrema=False)
    for body, s in zip(vp["bodies"], SERIES):
        body.set_facecolor(COLORS[s]); body.set_edgecolor("black")
        body.set_linewidth(0.8); body.set_alpha(1.0)
    vp["cmedians"].set_color("black"); vp["cmedians"].set_linewidth(0.8)

    ax.axhline(0, color="#999", linewidth=1.0, zorder=1)
    ax.set_xticks(range(len(SERIES))); ax.set_xticklabels(LABELS, fontsize=10)
    ax.set_ylabel("% change vs. real NTC", fontsize=11)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:+.0f}%" if abs(v) >= 0.5 else "0%"))
    ax.tick_params(axis="both", labelsize=9)
    ax.grid(axis="y", linewidth=0.5, alpha=0.35)


fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, f in zip(axes, FEATURES):
    plot_panel(ax, data[f["key"]], f["title"])

handles = [mpatches.Patch(facecolor=COLORS[s], edgecolor="black", linewidth=0.8, label=lab)
          for s, lab in zip(SERIES, LABELS)]
fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.05), frameon=False)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "counterfactual_traversal_violins.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "counterfactual_traversal_violins.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Mean, median, and n per series per feature — the %-change values plotted above.

In [ ]:
rows = []
for f in FEATURES:
    feat_data = data[f["key"]]
    base = float(np.mean(feat_data["real_ntc"]))
    for s in SERIES:
        vals = pct(feat_data[s], base)
        rows.append({
            "feature": f["key"], "series": s, "n": len(vals),
            "pct_mean": float(np.mean(vals)) if len(vals) else np.nan,
            "pct_median": float(np.median(vals)) if len(vals) else np.nan,
        })

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "counterfactual_traversal_violins_summary.csv", index=False)
summary